# DRfold Pilot10 — DRfold-only Structural Evaluation (10/10)

**CPU only.** This notebook computes **only DRfold** structural accuracy against experimental native structures: native-normalized TM-score, RMSD, alignment residues, and alignment coverage for the 10 frozen pilot RNA targets. It does **not** calculate or publish a five-model comparison.

Uses the matching previously verified **US-align v20260527** method (`-mol RNA -TMscore 1`) and native-observed-position mapping. Native coordinates are used for **evaluation only**.

Inputs are already on Google Drive at `MyDrive/RNA3D_Benchmark/DRfold/Evaluations/RNA3D_DRfold_Pilot10_Eval_Inputs_20261119.zip` (10 model predictions and native references). **No manual upload and no GPU required.**

**Success criterion:** `FINAL DRIVE VERIFICATION PASS: 10/10 DRfold scores intact`. No GitHub changes are made. Run the five-model comparison later as a separate step after inspecting DRfold results.


## 1. Mount Google Drive and load frozen evaluation input bundle (no manual upload)

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil, zipfile, hashlib

drive.mount('/content/drive')
DRIVE_EVAL=Path('/content/drive/MyDrive/RNA3D_Benchmark/DRfold/Evaluations')
EXPECTED='RNA3D_DRfold_Pilot10_Eval_Inputs_20261119.zip'
SOURCE_BUNDLE=DRIVE_EVAL/EXPECTED
assert SOURCE_BUNDLE.is_file(), f'Missing expected Drive file: {SOURCE_BUNDLE}; check signed-in account'
assert SOURCE_BUNDLE.stat().st_size==395773, f'Unexpected input bundle byte count: {SOURCE_BUNDLE.stat().st_size}'
BUNDLE=Path('/content')/EXPECTED
if BUNDLE.exists():
    assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==hashlib.sha256(SOURCE_BUNDLE.read_bytes()).hexdigest(), 'Existing local bundle differs from Drive bundle'
else:
    shutil.copy2(SOURCE_BUNDLE,BUNDLE)
with zipfile.ZipFile(BUNDLE,'r') as zin:
    assert zin.testzip() is None, 'Corrupt ZIP'
    assert 'manifest/drfold_10_preflight.csv' in zin.namelist()
    assert 'prior/pilot10_four_model_paired.csv' in zin.namelist()
print('✅ Loaded 10-target verified input bundle directly from Drive:',SOURCE_BUNDLE)
print('Bytes:',BUNDLE.stat().st_size,'SHA256:',hashlib.sha256(BUNDLE.read_bytes()).hexdigest())

## 2. Build pinned US-align v20260527

Uses the **same source commit** as the archived four-model evaluation. Requires internet; GPU is unnecessary.

In [ ]:
import subprocess, os
SRC=Path('/content/USalign_drfold_pilot10_20260527')
TAG='20260527'
EXPECTED_COMMIT='177cc8a2bbd3e2a6e9c5faaaa4ff5dfa1e6048f7'
if not SRC.exists():
    subprocess.run(['git','clone','--depth','1','--branch',TAG,'https://github.com/pylelab/USalign.git',str(SRC)],check=True)
actual=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
assert actual==EXPECTED_COMMIT, f'Unexpected US-align commit: {actual}'
USALIGN=SRC/'USalign'
if not USALIGN.exists():subprocess.run(['make'],cwd=SRC,check=True)
assert USALIGN.is_file() and os.access(USALIGN,os.X_OK)
print('Pinned US-align:',actual)
print('Binary:',USALIGN)

## 3. Evaluate the ten DRfold predictions (no other model scores)

In [ ]:
#!/usr/bin/env python3
"""DRfold-only pilot10 evaluation matching archived US-align v20260527 protocol.

Usage: python RNA3D_DRfold_Pilot10_Five_Model_Evaluation.py input.zip output_directory /path/to/USalign
Writes new immutable directory, never overwrites a completed evaluation.
"""
import csv, hashlib, io, json, math, os, re, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean

TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
MODEL_SHA='9586990c79e4ca488e5f967fcc8bd5b06cd60273'
USALIGN_SHA='177cc8a2bbd3e2a6e9c5faaaa4ff5dfa1e6048f7'

def sha(data):return hashlib.sha256(data).hexdigest()
def residue_list(pdb):
    records=[];seen=set()
    for line in pdb.decode(errors='replace').splitlines():
        if not line.startswith('ATOM  '): continue
        key=(line[21:22],line[22:26],line[26:27],line[17:20].strip())
        if key not in seen:
            seen.add(key);records.append({'key':key,'one':line[17:20].strip(),'pos':int(line[22:26])})
    return records

def score_parser(out,native_count):
    a=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([\d.eE+-]+)',out,re.I)
    if a:common,rmsd=int(a.group(1)),float(a.group(2))
    else:
        cm=re.search(r'Number of residues in common\s*=\s*(\d+)',out,re.I)
        rm=re.search(r'RMSD of the common residues\s*=\s*([\d.eE+-]+)',out,re.I)
        if not(cm and rm):raise ValueError('US-align aligned length or RMSD absent')
        common,rmsd=int(cm.group(1)),float(rm.group(1))
    scores=re.findall(r'TM-score\s*=\s*([\d.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',out,re.I)
    if not scores:raise ValueError('Missing native-normalized Structure_2 TM-score')
    tm=float(scores[-1]);assert 0<common<=native_count and math.isfinite(rmsd) and rmsd>=0
    assert math.isfinite(tm) and 0<=tm<=1.00001
    return {'tm_score':tm,'rmsd_angstrom':rmsd,'alignment_residues':common,'alignment_coverage':common/native_count}

# Parser self-test aligned with earlier pilot evaluation notebooks.
assert score_parser('Aligned length= 13, RMSD= 11.52, Seq_ID=n_identical/n_aligned= 1.000\nTM-score= 0.24331 (normalized by length of Structure_1: L=13, d0=0.40)\nTM-score= 0.24331 (normalized by length of Structure_2: L=13, d0=0.40)',13)=={'tm_score':0.24331,'rmsd_angstrom':11.52,'alignment_residues':13,'alignment_coverage':1.0}

def map_prediction(pdb,native_positions):
    residues=residue_list(pdb)
    # Model output residue order is required to match the complete benchmark target sequence.
    lookup={r['key']:i+1 for i,r in enumerate(residues)}
    mapped=[];serial=1
    for line in pdb.decode(errors='replace').splitlines():
        if not line.startswith(('ATOM  ','HETATM')):continue
        if line[16:17] not in (' ','A','1'):continue
        key=(line[21:22],line[22:26],line[26:27],line[17:20].strip())
        pos=lookup.get(key)
        if pos not in native_positions:continue
        mapped.append(line[:6]+f'{serial:5d}'+line[11:22]+f'{pos:4d}'+line[26:])
        serial+=1
    assert mapped,'Empty mapped prediction'
    return ('\n'.join(mapped+['TER','END'])+'\n').encode()

def write_csv(path,rows):
    with path.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0].keys()),lineterminator='\n');writer.writeheader();writer.writerows(rows)

def evaluate(bundle,dest,usalign):
    assert bundle.is_file() and usalign.is_file(), 'Missing input bundle or US-align binary'
    if dest.exists():raise FileExistsError(f'Never overwrite existing evaluation: {dest}')
    # Stage locally, no half-finished Drive evaluation. A failure leaves no FINALIZED marker.
    dest.mkdir(parents=True,exist_ok=False)
    (dest/'mapped').mkdir();(dest/'logs').mkdir()
    with zipfile.ZipFile(bundle) as z:
        assert z.testzip() is None,'Corrupt input bundle'
        names=z.namelist();assert all(not (n.startswith('/') or '..' in Path(n).parts) for n in names)
        def get(path):return z.read(path)
        manifest={r['target_id']:r for r in csv.DictReader(io.StringIO(get('manifest/drfold_10_preflight.csv').decode()))}
        assert set(manifest)==set(TARGETS)
        scores=[]
        for t in TARGETS:
            m=manifest[t]
            seq=''.join(x.strip() for x in get(f'predictions/{t}/{t}.fasta').decode().splitlines() if x.strip() and not x.startswith('>'))
            pred=get(f'predictions/{t}/DPR.pdb');native=get(f'native/{t}_native_eval.pdb')
            assert sha(pred)==m['prediction_sha256'] and sha(native)==m['native_sha256']
            assert sha(get(f'predictions/{t}/{t}.fasta'))==m['fasta_sha256']
            assert m['model_commit']==MODEL_SHA and set(seq)<=set('ACGU')
            predres=residue_list(pred);nres=residue_list(native)
            assert len(predres)==len(seq)==int(m['input_length'])
            assert ''.join(r['one'] for r in predres)==seq
            native_positions=[r['pos'] for r in nres]
            assert len(native_positions)==len(set(native_positions))==int(m['native_residues'])
            assert all(1<=r['pos']<=len(seq) and r['one']==seq[r['pos']-1] for r in nres)
            mapped=map_prediction(pred,set(native_positions))
            mapped_file=dest/'mapped'/f'{t}_drfold_mapped.pdb';mapped_file.write_bytes(mapped)
            native_file=dest/'mapped'/f'{t}_native_eval.pdb';native_file.write_bytes(native)
            cmd=[str(usalign),str(mapped_file),str(native_file),'-mol','RNA','-TMscore','1']
            r=subprocess.run(cmd,capture_output=True,text=True)
            output=r.stdout+('\n--- STDERR ---\n'+r.stderr if r.stderr else '')
            (dest/'logs'/f'{t}_usalign.txt').write_text(output)
            if r.returncode:raise RuntimeError(f'{t}: USalign failed: {r.returncode}: '+output[-700:])
            res=score_parser(r.stdout,len(nres))
            rec={'target_id':t,'model':'DRfold','input_length':len(seq),'native_residues':len(nres),'native_fraction':len(nres)/len(seq),
                 'prediction_sha256':sha(pred),'mapped_sha256':sha(mapped),'runtime_seconds':m['runtime_seconds'],
                 **res,'status':'scored'}
            scores.append(rec)
            print(f"{t:9s} TM={res['tm_score']:.5f} RMSD={res['rmsd_angstrom']:.2f} Å aligned={res['alignment_residues']}/{len(nres)}",flush=True)
    assert len(scores)==10 and {x['target_id'] for x in scores}==set(TARGETS)
    write_csv(dest/'drfold_scores.csv',scores)
    summary={'status':'complete','target_count':10,'model':'DRfold','drfold_model_commit':MODEL_SHA,'usalign_version':'20260527','usalign_source_commit':USALIGN_SHA,
             'method':'USalign mapped_prediction native_eval -mol RNA -TMscore 1 (Structure_2 native normalization)',
             'mean_tm':mean(float(r['tm_score']) for r in scores),
             'mean_rmsd_angstrom':mean(float(r['rmsd_angstrom']) for r in scores),
             'native_fraction_ge_0p8_count':sum(float(r['native_fraction'])>=0.8 for r in scores),
             'limitations':['exploratory n=10','different model input conditions','partially observed experimental structures','template scarcity and training overlap not established']}
    (dest/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
    filehashes={str(p.relative_to(dest)):sha(p.read_bytes()) for p in sorted(dest.rglob('*')) if p.is_file()}
    marker={'status':'complete','targets':10,'created_utc':datetime.now(timezone.utc).isoformat(),'files':filehashes}
    (dest/'FINALIZED.json').write_text(json.dumps(marker,indent=2)+'\n')
    assert all(sha((dest/name).read_bytes())==val for name,val in filehashes.items())
    print('FINAL VERIFICATION PASS: 10/10 DRfold scored in',dest)
    print(f"DRfold mean TM={summary['mean_tm']:.5f}; mean RMSD={summary['mean_rmsd_angstrom']:.2f} Å")
    return dest



## 4. Run DRfold-only scoring for all ten targets

In [ ]:
RESULT=Path('/content/drfold_pilot10_only_usalign20260527_20261119_v1')
result=evaluate(BUNDLE,RESULT,USALIGN)
assert (RESULT/'FINALIZED.json').is_file()
print('DRfold scores:',RESULT/'drfold_scores.csv')
print('DRfold summary:',RESULT/'summary.json')

## 5. Save DRfold-only results to Google Drive and create a backup ZIP

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from IPython.display import FileLink,display
assert (RESULT/'FINALIZED.json').is_file(), 'Evaluation not finalized; refusing to save'
ARCHIVE=Path('/content/RNA3D_DRfold_Pilot10_Only_Evaluation_Results.zip')
assert not ARCHIVE.exists(),'Do not overwrite a previous local archive'
with ZipFile(ARCHIVE,'x',compression=ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(RESULT.rglob('*')):
        if f.is_file():z.write(f,arcname=f'{RESULT.name}/{f.relative_to(RESULT)}')
with ZipFile(ARCHIVE) as z: assert z.testzip() is None
# Refuse overwrites: immutable previous evaluation runs must remain intact.
DEST=DRIVE_EVAL/RESULT.name
DRIVE_ARCHIVE=DRIVE_EVAL/ARCHIVE.name
assert DRIVE_EVAL.is_dir(), f'Expected Drive directory is missing: {DRIVE_EVAL}'
assert not DEST.exists() and not DRIVE_ARCHIVE.exists(), 'A matching evaluation already exists on Drive. Do not overwrite; inspect it.'
shutil.copytree(RESULT,DEST)
shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()==hashlib.sha256(DRIVE_ARCHIVE.read_bytes()).hexdigest(),'Drive ZIP copy mismatch'
for p in RESULT.rglob('*'):
    if p.is_file():
        remote=DEST/p.relative_to(RESULT)
        assert remote.is_file() and hashlib.sha256(remote.read_bytes()).hexdigest()==hashlib.sha256(p.read_bytes()).hexdigest(),f'Drive artifact mismatch: {p}'
print('✅ FINAL VERIFICATION PASS — evaluation saved and hashes verified on Google Drive')
print('Drive output:',DEST)
print('Drive ZIP:',DRIVE_ARCHIVE)
print('Optional local ZIP download:',ARCHIVE)
display(FileLink(str(ARCHIVE)))

## 6. Read-only final verification of DRfold scores saved on Drive

In [ ]:
import json,hashlib
DRIVE_RESULT=DRIVE_EVAL/'drfold_pilot10_only_usalign20260527_20261119_v1'
final=DRIVE_RESULT/'FINALIZED.json'
assert final.is_file(),f'Not finalized on Drive: {final}'
meta=json.loads(final.read_text())
assert meta.get('status')=='complete' and meta.get('targets')==10
assert (DRIVE_RESULT/'drfold_scores.csv').is_file()
for relative,digest in meta['files'].items():
    path=DRIVE_RESULT/relative
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest, f'Artifact missing/changed: {relative}'
print('FINAL DRIVE VERIFICATION PASS: 10/10 DRfold scores intact')
print('Report:',DRIVE_RESULT/'summary.json')